# Agentic Pipeline — Auto-Generated Notebook
**Pipeline ID:** `6450509b`  
**Status:** FAILED  
**Total time:** 265.11s


## Imports

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from imblearn.over_sampling import RandomOverSampler
import pandas as pd, numpy as np
from sklearn.cluster import KMeans
from sklearn.preprocessing import RobustScaler
from sklearn.compose import ColumnTransformer
_PIPELINE_ID = '6450509b'


## Step 1: Remove Missing Values

**LLM Reasoning:** Agent     : DynamicAgent[remove_missing_values] Status    : SUCCESS Duration  : 9.917s Timestamp : 2026-03-24T16:17:13.686436+00:00 Input     : DataFrame(20640x10) nulls=207


In [ ]:
# Verify df is a DataFrame
assert isinstance(df, pd.DataFrame), "df must be a pandas DataFrame"
# Identify columns with missing values
missing_counts = df.isnull().sum()
cols_with_missing = missing_counts[missing_counts > 0].index.tolist()
# Impute numeric columns with median (robust to skewness)
for col in cols_with_missing:
    if pd.api.types.is_numeric_dtype(df[col]):
        median_val = df[col].median()
        df[col].fillna(median_val, inplace=True)
    else:
        # For non‑numeric (unlikely here) use mode
        mode_val = df[col].mode().iloc[0]
        df[col].fillna(mode_val, inplace=True)
# Validation: ensure no missing values remain
assert df.isnull().sum().sum() == 0, "Missing values still present after imputation"


## Step 2: Handle Class Imbalance

**LLM Reasoning:** Agent     : DynamicAgent[handle_class_imbalance] Status    : SUCCESS Duration  : 13.013s Timestamp : 2026-03-24T16:17:26.716061+00:00 Input     : DataFrame(20640x10) nulls=0


In [ ]:
# Verify df is a DataFrame
assert isinstance(df, pd.DataFrame), "df must be a pandas DataFrame"
# ----- Encode target (categorical) temporarily -----
le = LabelEncoder()
# Preserve original column for later restoration
df['_target_enc'] = le.fit_transform(df['ocean_proximity'])
# Features (all columns except original target) and target
X = df.drop(columns=['ocean_proximity'])
y = df['_target_enc']
# ----- Oversample minority classes to match the majority class -----
ros = RandomOverSampler(sampling_strategy='auto', random_state=42)
X_res, y_res = ros.fit_resample(X, y)
# ----- Reconstruct DataFrame with original target labels -----
# Decode the target back to original string labels
original_target = le.inverse_transform(y_res)
# Build the balanced DataFrame
balanced_df = X_res.copy()
balanced_df['ocean_proximity'] = original_target
# Drop the temporary encoded column if it was carried over
if '_target_enc' in balanced_df.columns:
    balanced_df = balanced_df.drop(columns=['_target_enc'])
# Replace the original df with the balanced version
df = balanced_df.reset_index(drop=True)
# ----- Validation check -----
class_counts = df['ocean_proximity'].value_counts()
print('Class distribution after balancing:')
print(class_counts)


## Step 3: Encode Categorical

**LLM Reasoning:** Agent     : DynamicAgent[encode_categorical] Status    : SUCCESS Duration  : 10.785s Timestamp : 2026-03-24T16:17:37.520030+00:00 Input     : DataFrame(45680x10) nulls=0


In [ ]:
# Ensure df is a DataFrame
assert isinstance(df, pd.DataFrame)
# Detect categorical columns (object or category dtype)
cat_cols = df.select_dtypes(include=['object', 'category']).columns.tolist()
# Dictionary to keep encoders for possible inverse‑transform later
categorical_encoders = {}
for col in cat_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])
    categorical_encoders[col] = le
# Validation: all encoded columns should now be integer dtype and retain original cardinality
assert all(pd.api.types.is_integer_dtype(df[col].dtype) for col in cat_cols), "Encoding failed: non‑integer dtype detected"
assert all(df[col].nunique() == len(categorical_encoders[col].classes_) for col in cat_cols), "Encoding failed: cardinality mismatch"


## Step 4: Handle Skewness

**LLM Reasoning:** Agent     : DynamicAgent[handle_skewness] Status    : SUCCESS Duration  : 14.686s Timestamp : 2026-03-24T16:18:06.028522+00:00 Input     : DataFrame(45680x10) nulls=0


In [ ]:
# Verify df is a DataFrame
assert isinstance(df, pd.DataFrame), 'df must be a pandas DataFrame'
# Columns identified as highly skewed in the schema
skewed_cols = [
    "total_rooms",
    "total_bedrooms",
    "population",
    "households",
    "median_income"
]
# Apply log1p transformation to each skewed column
for col in skewed_cols:
    # Ensure the column exists and contains only non‑negative values
    assert col in df.columns, f"Column {col} not found in df"
    assert (df[col] >= 0).all(), f"Column {col} contains negative values, cannot apply log1p"
    df[col] = np.log1p(df[col])
# Validation: check that skewness has been reduced for each transformed column
reduced = {}
for col in skewed_cols:
    original_skew = {
        "total_rooms": 4.2789,
        "total_bedrooms": 3.6176,
        "population": 4.5993,
        "households": 3.5476,
        "median_income": 1.9887
    }[col]
    new_skew = df[col].skew()
    reduced[col] = {
        "original": original_skew,
        "new": new_skew,
        "reduced": abs(new_skew) < abs(original_skew)
    }
    assert reduced[col]["reduced"], f"Skewness not reduced for {col}: {original_skew} -> {new_skew}"
# Optional: expose the validation dictionary for downstream steps
skewness_reduction_report = reduced


## Step 5: Feature Engineering

**LLM Reasoning:** Agent     : DynamicAgent[feature_engineering] Status    : SUCCESS Duration  : 27.419s Timestamp : 2026-03-24T16:18:33.464682+00:00 Input     : DataFrame(45680x10) nulls=0


In [ ]:
# Create a copy to avoid modifying original unintentionally
_df = df.copy()
# Ratio / per‑capita features
_df['rooms_per_household'] = _df['total_rooms'] / _df['households']
_df['bedrooms_per_room'] = _df['total_bedrooms'] / _df['total_rooms']
_df['population_per_household'] = _df['population'] / _df['households']
_df['bedrooms_per_household'] = _df['total_bedrooms'] / _df['households']
_df['rooms_per_person'] = _df['total_rooms'] / _df['population']
# Interaction between geographic coordinates
_df['lat_long_interaction'] = _df['latitude'] * _df['longitude']
# Log‑transform median_income (already handled earlier but safe to keep)
_df['log_median_income'] = np.log1p(_df['median_income'])
# Simple geo‑clustering to capture regional patterns
kmeans = KMeans(n_clusters=5, random_state=42, n_init='auto')
_df['geo_cluster'] = kmeans.fit_predict(_df[['latitude', 'longitude']])
# Replace original df with engineered version
df = _df


## Step 6: Normalize Features

**LLM Reasoning:** Agent     : DynamicAgent[normalize_features] Status    : SUCCESS Duration  : 19.982s Timestamp : 2026-03-24T16:18:53.465699+00:00 Input     : DataFrame(45680x18) nulls=0


In [ ]:
# Identify target and numeric feature columns
TARGET_COL = 'geo_cluster'
numeric_cols = df.select_dtypes(include=['float64', 'int64', 'int32']).columns.tolist()
numeric_cols = [c for c in numeric_cols if c != TARGET_COL]
# Build transformer: RobustScaler for all numeric features, passthrough for the rest (including target)
ct = ColumnTransformer(
    [('num', RobustScaler(), numeric_cols)],
    remainder='passthrough'
)
# Apply transformation
transformed = ct.fit_transform(df)
# Re‑assemble DataFrame with original column order
# ColumnTransformer returns numeric cols first, then passthrough cols in their original order
passthrough_cols = [c for c in df.columns if c not in numeric_cols]
new_columns = numeric_cols + passthrough_cols
df = pd.DataFrame(transformed, columns=new_columns)
# Preserve original dtype for the target (int)
df[TARGET_COL] = df[TARGET_COL].astype(int)
# Validation: ensure no NaNs were introduced and median of scaled features is ~0
assert not df.isnull().any().any(), "NaNs introduced during scaling"
assert np.allclose(df[numeric_cols].median().abs(), 0, atol=1e-6), "Median of scaled features not close to 0"


## Step 7: Select And Train Models [Failed]

**LLM Reasoning:** Agent    : DynamicAgent[select_and_train_models] Status   : FAILED Duration : 65.559s This step failed. No code was executed. Re-run the pipeline to regenerate.


In [ ]:
# raise RuntimeError('Step 'select_and_train_models' failed during pipeline run')
# SAVE OUTPUTS
import os
os.makedirs('outputs', exist_ok=True)
# Save cleaned dataset
df.to_csv('outputs/cleaned_data.csv', index=False, encoding='utf-8')
print(f'Cleaned data saved -> outputs/cleaned_data.csv ({df.shape[0]} rows x {df.shape[1]} cols)')
# Save trained model
import joblib
if 'trained_model' in dir():
    joblib.dump(trained_model, 'outputs/model.pkl')
    print(f'Model saved -> outputs/model.pkl ({type(trained_model).__name__})')
elif 'trained_models' in dir() and isinstance(trained_models, dict):
    # Save best model (first in dict) and all models
    best_name, best_model = next(iter(trained_models.items()))
    joblib.dump(best_model, 'outputs/model.pkl')
    print(f'Best model saved -> outputs/model.pkl ({best_name}: {type(best_model).__name__})')
    # Save all models
    for name, model in trained_models.items():
        joblib.dump(model, f'outputs/model_{name}.pkl')
    print(f'All {len(trained_models)} models saved to outputs/')
else:
    print('WARNING: No trained model found to save')
print('All outputs saved to outputs/ directory')


## Saved Outputs

In [ ]:
# ── Load pipeline outputs ────────────────────────────
import pandas as pd

# Cleaned dataset
df_clean = pd.read_csv('outputs/cleaned_data.csv', encoding='utf-8')
print(f'Cleaned data: {df_clean.shape[0]} rows x {df_clean.shape[1]} cols')
df_clean.head()
